<a href="https://colab.research.google.com/github/aiaanad/metopt26/blob/main/lab4/%D0%A2%D0%B5%D0%BC%D0%B0_4_%D0%94%D0%B0%D0%BD%D0%B8%D0%BB%D0%BE%D0%B2%D0%B0_%D0%90_%D0%92.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from enum import Enum
import operator

class Operation(Enum):
    MAX = "max"
    MIN = "min"

    @property
    def neutral(self) -> float:
        return float('-inf') if self == Operation.MAX else float('inf')

    @property
    def better(self):
        return operator.gt if self == Operation.MAX else operator.lt

In [11]:
from dataclasses import dataclass, field
from typing import List, Callable, Optional

@dataclass
class DPInput:
    data: List[List[float]]
    op: Operation = Operation.MAX
    limit_t: Optional[int] = None
    filters: List[Callable[[int], bool]] = field(default_factory=list)

In [16]:
from typing import Optional


class DPSolver:
    def __init__(self, task: 'DPInput'):
        self.task = task
        self.m = len(task.data)
        self.n = len(task.data[0]) - 1

        self.use_k: bool = task.limit_t is not None
        self.t: Optional[int] = task.limit_t

        self.v = None
        self.choices = None


    def _local_ok(self, i: int, u: int) -> bool:
        """локальный фильтр для объектов"""
        if i < len(self.task.filters):
            return self.task.filters[i](u)
        return True

    def _better(self, a, b) -> bool:
        """a лучше b согласно OP (max/min)"""
        return self.task.op.better(a, b)

    def _pick(self, best_val, best_u, val, u):
        """обновление лучшего кандидата"""
        if best_u is None or self._better(val, best_val):
            return u, val
        return best_u, best_val


    def solve(self):
        m, n = self.m, self.n
        OP = self.task.op

        if self.use_k:
            self._solve_with_limit()
        else:
            self._solve_without_limit()

        return self.v[0][n][0] if self.use_k else self.v[0][n]


    def _solve_without_limit(self):
        """2D-ДП: V[i][s]"""
        m, n, OP = self.m, self.n, self.task.op

        self.v = [[OP.neutral] * (n + 1) for _ in range(m + 1)]
        self.choices = [[None] * (n + 1) for _ in range(m)]

        for s in range(n + 1):
            self.v[m][s] = 0

        for i in range(m - 1, -1, -1):
            for s in range(n + 1):
                best_u, best_val = None, OP.neutral
                for u in range(s + 1):
                    if not self._local_ok(i, u):
                        continue
                    current = self.task.data[i][u] + self.v[i + 1][s - u]
                    best_u, best_val = self._pick(best_val, best_u, current, u)
                self.v[i][s] = best_val
                self.choices[i][s] = best_u


    def _solve_with_limit(self):
        """3D-ДП: V[i][s][k], где k число уже задействованных объектов"""
        m, n, t, OP = self.m, self.n, self.t, self.task.op

        self.v = [[[OP.neutral] * (t + 1) for _ in range(n + 1)] for _ in range(m + 1)]
        self.choices = [[[None] * (t + 1) for _ in range(n + 1)] for _ in range(m)]

        for s in range(n + 1):
            for k in range(t + 1):
                self.v[m][s][k] = 0

        for i in range(m - 1, -1, -1):
            for s in range(n + 1):
                for k in range(t + 1):
                    best_u, best_val = None, OP.neutral
                    for u in range(s + 1):
                        if not self._local_ok(i, u):
                            continue
                        nz = 1 if u > 0 else 0
                        k_new = k + nz
                        if k_new > t:
                            continue
                        current = self.task.data[i][u] + self.v[i + 1][s - u][k_new]
                        best_u, best_val = self._pick(best_val, best_u, current, u)
                    self.v[i][s][k] = best_val
                    self.choices[i][s][k] = best_u


    def reconstruct(self):
        if self.choices is None:
            raise RuntimeError("Сначала вызовите solve()")

        s = self.n
        k = 0
        solution = []
        for i in range(self.m):
            if self.use_k:
                u = self.choices[i][s][k]
                k += 1 if u > 0 else 0
            else:
                u = self.choices[i][s]
            solution.append(u)
            s -= u
        return solution

In [17]:
data = [
    [0, 5, 9, 13, 16, 18, 20, 21, 22, 23, 24], # А
    [0, 3, 8, 14, 18, 21, 23, 24, 25, 26, 27], # Б
    [0, 4, 10, 13, 15, 17, 19, 20, 21, 22, 23] # В
]

task = DPInput(
    data=data,
    op=Operation.MAX,
    limit_t=2,
    filters=[lambda u: True, lambda u: True, lambda u: True]
)
solver = DPSolver(task)
max_value = solver.solve()
solution = solver.reconstruct()
print(f"Максимум: {max_value}, Решение: {solution}")

Максимум: 39, Решение: [4, 6, 0]
